In [1]:
import cv2
import os
import numpy as np
import torch
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.preprocessing import StandardScaler, LabelEncoder, normalize
from sklearn.svm import SVC
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA   #dimension reduce
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import entropy

# EfficientNet B0
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)
model.classifier = torch.nn.Identity()
model.eval()
transform = transforms.Compose([transforms.ToPILImage(), transforms.Resize((224,224)), transforms.ToTensor()])

def extract_deep_features(img):
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor = transform(img).unsqueeze(0)
    with torch.no_grad():
        features = model(tensor)
    return features.numpy().flatten()

def extract_handcrafted_features(img):
    img = cv2.resize(img,(256,256))
    hsv = cv2.cvtColor(img,cv2.COLOR_BGR2HSV)
    s = hsv[:,:,1]
    _,mask = cv2.threshold(s,30,255,cv2.THRESH_BINARY)
    kernel = np.ones((5,5),np.uint8)
    mask = cv2.morphologyEx(mask,cv2.MORPH_CLOSE,kernel)
    contours,_ = cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if len(contours)==0:
        return None
    cnt=max(contours,key=cv2.contourArea)
    area=cv2.contourArea(cnt)
    perimeter=cv2.arcLength(cnt,True)
    circularity=(4*np.pi*area)/(perimeter**2+1e-6)
    hull=cv2.convexHull(cnt)
    solidity=area/(cv2.contourArea(hull)+1e-6)
    x,y,w,h=cv2.boundingRect(cnt)
    aspect_ratio=w/h
    extent=area/(w*h+1e-6)
    R,G,B=cv2.split(img)
    R_mean,R_std=cv2.meanStdDev(R,mask=mask)
    G_mean,G_std=cv2.meanStdDev(G,mask=mask)
    B_mean,B_std=cv2.meanStdDev(B,mask=mask)
    H,S,V=cv2.split(hsv)
    H_mean,H_std=cv2.meanStdDev(H,mask=mask)
    S_mean,S_std=cv2.meanStdDev(S,mask=mask)
    V_mean,V_std=cv2.meanStdDev(V,mask=mask)
    lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
    L,A,B_lab=cv2.split(lab)
    L_mean,L_std=cv2.meanStdDev(L,mask=mask)
    a_mean,a_std=cv2.meanStdDev(A,mask=mask)
    b_mean,b_std=cv2.meanStdDev(B_lab,mask=mask)
    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    gray_masked=cv2.bitwise_and(gray,gray,mask=mask)
    laplacian=cv2.Laplacian(gray_masked,cv2.CV_64F).var()
    glcm=graycomatrix(gray_masked,[1],[0],256,symmetric=True,normed=True)
    contrast=graycoprops(glcm,'contrast')[0,0]
    energy=graycoprops(glcm,'energy')[0,0]
    homogeneity=graycoprops(glcm,'homogeneity')[0,0]
    hist=cv2.calcHist([gray_masked],[0],mask,[256],[0,256])
    hist_norm=hist/hist.sum()
    ent=entropy(hist_norm.flatten())
    dark_pixel_ratio=np.sum(gray_masked<50)/(np.sum(mask>0)+1e-6)
    return [R_mean[0][0],G_mean[0][0],B_mean[0][0],R_std[0][0],G_std[0][0],B_std[0][0],
            H_mean[0][0],S_mean[0][0],V_mean[0][0],H_std[0][0],S_std[0][0],V_std[0][0],
            L_mean[0][0],L_std[0][0],a_mean[0][0],a_std[0][0],b_mean[0][0],b_std[0][0],
            laplacian,contrast,energy,homogeneity,ent,
            area,perimeter,circularity,solidity,aspect_ratio,extent,dark_pixel_ratio]

# NEW DATASET PATH
dataset_path = r"C:\Users\KIIT0001\Downloads\archive (9)\vegetable_Dataset"

data = []
labels = []

for folder in os.listdir(dataset_path):

    folder_path = os.path.join(dataset_path, folder)

    if not os.path.isdir(folder_path):
        continue

    label = folder

    for img_name in os.listdir(folder_path):

        img_path = os.path.join(folder_path, img_name)

        img = cv2.imread(img_path)

        if img is None:
            continue

        hand_feat = extract_handcrafted_features(img)

        if hand_feat is None:
            continue

        deep_feat = extract_deep_features(img)

        fused_features = np.concatenate((hand_feat, deep_feat))

        data.append(fused_features)
        labels.append(label)

# Train SVM
X_raw = np.array(data)
y_raw = np.array(labels)

le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

X_norm = normalize(X_scaled)

# ✅ APPLY PCA
pca = PCA(n_components=200)
X_pca = pca.fit_transform(X_norm)

svm_model = SVC(kernel='rbf', probability=True)
svm_model.fit(X_pca, y_encoded)

def optimize_and_identify(img, svm_model, scaler):

    hand_feat = extract_handcrafted_features(img)
    deep_feat = extract_deep_features(img)

    fused = np.concatenate((hand_feat, deep_feat))

    fused_scaled = scaler.transform([fused])
    fused_norm = normalize(fused_scaled)

    # ✅ APPLY PCA
    fused_pca = pca.transform(fused_norm)

    pred_class = svm_model.predict(fused_pca)[0]
    pred_label = le.inverse_transform([pred_class])[0].lower()

    if "potato" in pred_label: item = "Potato"
    elif "cucumber" in pred_label: item = "Cucumber"
    elif "capsicum" in pred_label: item = "Capsicum"
    elif "banana" in pred_label: item = "Banana"
    elif "apple" in pred_label: item = "Apple"
    else: item = "Unknown"

    return fused_norm[0], item


def compute_freshness_cosine(fused_norm, item, X, y):

    idx = [i for i,lbl in enumerate(y) if item.lower() in lbl.lower()]

    item_features = X[idx]
    item_labels = [y[i] for i in idx]

    sims = cosine_similarity([fused_norm], item_features)[0]

    freshness_values = np.array([1 if "fresh" in lbl.lower() else 0 for lbl in item_labels])

    sims = np.maximum(sims,0)

    score = np.sum(sims*freshness_values)/(np.sum(sims)+1e-8)*100

    if score<25: category="Fully Rotten"
    elif score<50: category="Rotten"
    elif score<75: category="Fresh"
    else: category="Fully Fresh"

    return round(score,2), category


# Test images

img_path1 = r"C:\Users\KIIT0001\Downloads\captivating-close-up-image-showcases-intricate-decay-process-single-rotten-apple-against-pristine-white-background-391478811.webp"
img_path2 = r"C:\Users\KIIT0001\Downloads\istockphoto-184952247-612x612.jpg"

for img_path in [img_path1, img_path2]:

    img = cv2.imread(img_path)

    if img is None:
        print("Image not found:", img_path)
        continue

    fused_norm, item = optimize_and_identify(img, svm_model, scaler)

    score, category = compute_freshness_cosine(fused_norm, item, X_norm, labels)

    print("Item Detected:", item)
    print("Freshness Score:", score,"%")
    print("Category:", category)

Item Detected: Apple
Freshness Score: 48.31 %
Category: Rotten
Item Detected: Banana
Freshness Score: 29.24 %
Category: Rotten
